# 01 — Data Preprocessing & Exploratory Data Analysis

**Objectives**
- Inspect and clean the Telco customer churn dataset.
- Detect outliers in **Monthly Charge(s)**, **Total Charges**, and **Tenure Months**.
- Treat charge outliers using IQR-based capping while preserving the original values.
- Keep valid tenure extremes; flag them rather than automatically removing/capping them.
- Explore churn rate, distributions, correlations, and geographic patterns by City and State.

> The notebook preserves the original charge columns and creates capped versions (`*_Capped`). This makes the treatment transparent and allows you to compare results before and after treatment.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

# Update this path if your Excel file is stored somewhere else.
FILE_PATH = "/content/Telco_customer_churn.xlsx"
df = pd.read_excel(FILE_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
df.info()

## 2. Understand the data

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes.to_frame("Data type"))

print("\\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("Missing values"))

print(f"\\nDuplicate rows: {df.duplicated().sum():,}")

constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
print("Constant columns:", constant_cols)

for col in ["Count", "Country"]:
    if col in df.columns:
        print(f"\\n{col} — unique values: {df[col].nunique(dropna=False)}")
        display(df[col].value_counts(dropna=False).head(10))

## 3. Clean column names, text, missing values, and data types

Text columns are stripped **before** converting numeric fields. This is important because values such as `" "` in `Total Charges` should become missing values, not remain text.

In [ ]:
# Remove leading/trailing spaces from column names and text values.
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Support either singular or plural Monthly Charge column naming.
def find_column(possible_names, columns):
    lookup = {c.lower().strip(): c for c in columns}
    for name in possible_names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

monthly_col = find_column(["Monthly Charge", "Monthly Charges"], df.columns)
total_col = find_column(["Total Charges", "Total Charge"], df.columns)
tenure_col = find_column(["Tenure Months", "Tenure Month"], df.columns)
churn_col = find_column(["Churn Label", "Churn"], df.columns)
city_col = find_column(["City"], df.columns)
state_col = find_column(["State"], df.columns)

required = {
    "Monthly Charge(s)": monthly_col,
    "Total Charges": total_col,
    "Tenure Months": tenure_col,
    "Churn Label": churn_col
}
missing_required = [name for name, col in required.items() if col is None]
if missing_required:
    raise KeyError(f"Could not find these required columns: {missing_required}")

print("Detected columns:", required)

# Convert target numeric columns safely.
for col in [monthly_col, total_col, tenure_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove exact duplicate rows.
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df):,} duplicate rows.")

print("\\nMissing values after cleaning:")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("Missing values"))

print("\\nNumeric summary:")
display(df[[monthly_col, total_col, tenure_col]].describe().T)

## 4. Detect and treat outliers

### Method
- Use the **1.5 × IQR rule** to identify statistical outliers.
- **Monthly Charges** and **Total Charges:** create IQR-capped columns. Original values remain unchanged for auditability.
- **Tenure Months:** flag IQR outliers, but do not cap valid low/high tenure values. Tenure is naturally bounded (typically 0–72 months), and extreme values may be genuine customer histories.
- Check impossible domain values separately (negative charges or tenure outside the dataset's valid range).

IQR outliers are not automatically errors. In particular, high Total Charges can be legitimate because they accumulate over a customer's tenure.

In [ ]:
outlier_cols = [monthly_col, total_col, tenure_col]
outlier_rows = []

iqr_bounds = {}

for col in outlier_cols:
    series = df[col].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)

    iqr_bounds[col] = (lower, upper)
    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower bound": lower,
        "Upper bound": upper,
        "Outlier count": int(mask.sum()),
        "Outlier percentage": 100 * mask.mean()
    })

outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)

# Visualize the three variables before treatment.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(f"{col} — before treatment")
plt.tight_layout()
plt.show()

# Preserve raw data and add outlier flags.
for col in outlier_cols:
    lower, upper = iqr_bounds[col]
    df[f"{col}_IQR_Outlier"] = ((df[col] < lower) | (df[col] > upper)).astype("int8")

# Treat charge outliers by winsorizing/capping at IQR limits.
# The original columns are retained; use the *_Capped columns when a capped
# version is appropriate for modelling or sensitivity analysis.
for col in [monthly_col, total_col]:
    lower, upper = iqr_bounds[col]
    df[f"{col}_Capped"] = df[col].clip(lower=lower, upper=upper)

# Do not IQR-cap valid tenure observations. Check domain validity instead.
# This dataset commonly records tenure from 0 to 72 months.
df[f"{tenure_col}_Domain_Invalid"] = (
    (df[tenure_col] < 0) | (df[tenure_col] > 72)
).astype("int8")

print("Treatment summary:")
for col in [monthly_col, total_col]:
    capped_col = f"{col}_Capped"
    changed = (df[col].notna() & df[capped_col].notna() & (df[col] != df[capped_col])).sum()
    print(f"{col}: {changed:,} values capped in {capped_col}")

print(f"Tenure IQR outliers flagged: {df[f'{tenure_col}_IQR_Outlier'].sum():,}")
print(f"Tenure values outside 0–72 months: {df[f'{tenure_col}_Domain_Invalid'].sum():,}")

In [ ]:
# Compare charge distributions before and after capping.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df[monthly_col].dropna(), kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Monthly Charges — original")

sns.histplot(df[f"{monthly_col}_Capped"].dropna(), kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Monthly Charges — capped")

sns.histplot(df[total_col].dropna(), kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Total Charges — original")

sns.histplot(df[f"{total_col}_Capped"].dropna(), kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Total Charges — capped")

plt.tight_layout()
plt.show()

# Compare summary statistics.
comparison = pd.DataFrame({
    "Monthly original": df[monthly_col].describe(),
    "Monthly capped": df[f"{monthly_col}_Capped"].describe(),
    "Total original": df[total_col].describe(),
    "Total capped": df[f"{total_col}_Capped"].describe()
})
display(comparison)

## 5. Exploratory Data Analysis (EDA)
### 5.1 Churn distribution and churn rate

In [ ]:
# Normalize common Yes/No labels for consistent calculations.
churn_values = df[churn_col].astype("string").str.strip().str.lower()
print("Churn label counts:")
display(df[churn_col].value_counts(dropna=False))

yes_labels = {"yes", "1", "true", "churn"}
no_labels = {"no", "0", "false", "not churn"}

churn_binary = churn_values.map(lambda x: 1 if x in yes_labels else (0 if x in no_labels else np.nan))
if churn_binary.notna().any():
    overall_churn_rate = churn_binary.mean() * 100
    print(f"Overall churn rate: {overall_churn_rate:.2f}%")
else:
    print("Could not automatically calculate churn rate. Check the churn label categories.")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=df, x=churn_col, order=df[churn_col].value_counts().index, ax=axes[0])
axes[0].set_title("Customer count by churn status")
axes[0].tick_params(axis="x", rotation=20)

churn_pct = df[churn_col].value_counts(normalize=True).mul(100)
sns.barplot(x=churn_pct.index.astype(str), y=churn_pct.values, ax=axes[1])
axes[1].set_title("Churn label distribution (%)")
axes[1].set_ylabel("Percentage")
axes[1].set_xlabel(churn_col)
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### 5.2 Numeric distributions and comparison by churn

In [ ]:
# Histograms for the three requested variables.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, [monthly_col, total_col, tenure_col]):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# Compare distributions between churn groups.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, [monthly_col, total_col, tenure_col]):
    sns.boxplot(data=df, x=churn_col, y=col, ax=ax)
    ax.set_title(f"{col} by churn status")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# Mean and median by churn category.
display(
    df.groupby(churn_col)[[monthly_col, total_col, tenure_col]]
      .agg(["count", "mean", "median"])
      .round(2)
)

### 5.3 Correlation analysis

In [ ]:
# Include numeric features only. Encode churn as 0/1 when possible.
corr_df = df.select_dtypes(include=np.number).copy()
if churn_binary.notna().all():
    corr_df["Churn_Binary"] = churn_binary.astype(int)

corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            linewidths=0.4, annot_kws={"size": 8})
plt.title("Correlation matrix — numeric features")
plt.tight_layout()
plt.show()

if "Churn_Binary" in corr.columns:
    print("Correlation with churn (sorted):")
    display(corr["Churn_Binary"].drop("Churn_Binary").sort_values(ascending=False).to_frame("Correlation with churn"))

### 5.4 Geographic patterns — State and City

The analysis below calculates customer counts and churn rates by State and City. City-level results are restricted to cities with a minimum number of customers so that very small groups do not dominate the chart.

In [ ]:
if churn_binary.notna().any():
    geo_df = df.copy()
    geo_df["_Churn_Binary"] = churn_binary

    # State-level customer volume and churn rate.
    if state_col is not None:
        state_summary = (
            geo_df.dropna(subset=[state_col])
                  .groupby(state_col)
                  .agg(Customers=("_Churn_Binary", "size"),
                       Churned=("_Churn_Binary", "sum"),
                       Churn_Rate=("_Churn_Binary", "mean"))
                  .reset_index()
        )
        state_summary["Churn_Rate"] *= 100
        state_summary = state_summary.sort_values("Churn_Rate", ascending=False)

        print("State-level churn summary:")
        display(state_summary.head(20).round(2))

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        top_states_volume = state_summary.nlargest(15, "Customers").sort_values("Customers")
        sns.barplot(data=top_states_volume, y=state_col, x="Customers", ax=axes[0])
        axes[0].set_title("Top states by customer count")

        top_states_churn = state_summary.nlargest(15, "Churn_Rate").sort_values("Churn_Rate")
        sns.barplot(data=top_states_churn, y=state_col, x="Churn_Rate", ax=axes[1])
        axes[1].set_title("States with highest churn rate")
        axes[1].set_xlabel("Churn rate (%)")
        plt.tight_layout()
        plt.show()
    else:
        print("State column not found; skipping state-level analysis.")

    # City-level analysis. Use a minimum group size to reduce unstable rates.
    if city_col is not None:
        city_summary = (
            geo_df.dropna(subset=[city_col])
                  .groupby(city_col)
                  .agg(Customers=("_Churn_Binary", "size"),
                       Churned=("_Churn_Binary", "sum"),
                       Churn_Rate=("_Churn_Binary", "mean"))
                  .reset_index()
        )
        city_summary["Churn_Rate"] *= 100
        MIN_CUSTOMERS_PER_CITY = 30
        city_filtered = city_summary[
            city_summary["Customers"] >= MIN_CUSTOMERS_PER_CITY
        ].copy()

        print(f"City-level churn summary (cities with at least {MIN_CUSTOMERS_PER_CITY} customers):")
        display(city_filtered.sort_values("Churn_Rate", ascending=False).head(20).round(2))

        if not city_filtered.empty:
            top_cities = city_filtered.nlargest(15, "Churn_Rate").sort_values("Churn_Rate")
            plt.figure(figsize=(10, 7))
            sns.barplot(data=top_cities, y=city_col, x="Churn_Rate")
            plt.title(f"Cities with highest churn rate (minimum {MIN_CUSTOMERS_PER_CITY} customers)")
            plt.xlabel("Churn rate (%)")
            plt.tight_layout()
            plt.show()
        else:
            print("No cities met the minimum customer threshold. Reduce MIN_CUSTOMERS_PER_CITY if appropriate.")
    else:
        print("City column not found; skipping city-level analysis.")
else:
    print("Geographic churn analysis skipped because churn labels could not be mapped to Yes/No.")

## 6. Optional: inspect churn rate by contract or service categories

In [ ]:
# Useful categorical comparisons, if these columns exist.
candidate_cats = ["Contract", "Internet Service", "Payment Method", "Paperless Billing"]
available_cats = [c for c in candidate_cats if c in df.columns]

if churn_binary.notna().any():
    for col in available_cats:
        temp = df[[col]].copy()
        temp["_Churn_Binary"] = churn_binary
        summary = (temp.groupby(col, dropna=False)["_Churn_Binary"]
                       .agg(Customers="size", Churn_Rate="mean")
                       .reset_index())
        summary["Churn_Rate"] *= 100

        print(f"Churn rate by {col}:")
        display(summary.sort_values("Churn_Rate", ascending=False).round(2))

        plt.figure(figsize=(9, 4))
        sns.barplot(data=summary, x=col, y="Churn_Rate")
        plt.title(f"Churn rate by {col}")
        plt.ylabel("Churn rate (%)")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()

## 7. Final checks and save the cleaned dataset

The export includes:
- cleaned original columns;
- IQR outlier flags;
- capped charge columns;
- tenure domain-validity flag.

For modelling, choose deliberately whether to use the original charge variables or their capped versions. Avoid using both versions of the same variable in one model unless you have a specific reason.

In [ ]:
print("Final dataset shape:", df.shape)
print("\\nRemaining missing values in key numeric columns:")
display(df[[monthly_col, total_col, tenure_col]].isna().sum().to_frame("Missing values"))

print("\\nOutlier/domain flags:")
flag_cols = [c for c in df.columns if c.endswith("_IQR_Outlier") or c.endswith("_Domain_Invalid")]
display(df[flag_cols].sum().to_frame("Flagged rows"))

CLEAN_PATH = "telco_cleaned_eda.csv"
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned dataset to: {CLEAN_PATH}")